# Apache Polaris — RBAC Privilege Verification

**Date:** 2026-05-15  
**Reference:** https://polaris.apache.org/releases/1.3.0/managing-security/access-control/

## 학습 목표
- 각 Principal의 실제 권한 체인 확인
- Privilege scope (catalog/namespace/table/view) 동작 검증
- 권한 있는 작업 vs 권한 없는 작업 실제 확인
- Grant resolution 흐름 직접 확인

## 핵심 규칙
- Privilege는 Catalog Role에 부여됨
- Principal Role에 직접 부여 불가
- Catalog scope = 카탈로그 내 모든 해당 타입 객체에 적용
- Scope ≠ 권한 상속 (TABLE_READ_DATA at catalog ≠ NAMESPACE_LIST)

In [1]:
# ============================================================
# Prerequisites
# uv add requests pandas sqlalchemy psycopg2-binary
# ============================================================
import subprocess, sys, os
REQUIRED = ['requests', 'pandas', 'sqlalchemy', 'psycopg2-binary']
def install(pkg):
    r = subprocess.run(['uv', 'add', pkg], capture_output=True, text=True, cwd=os.getcwd())
    if r.returncode == 0: return '✅', 'uv'
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], capture_output=True, text=True)
    return ('✅', 'pip') if r.returncode == 0 else ('❌', 'failed')
for pkg in REQUIRED:
    s, m = install(pkg)
    print(f'  {s} {pkg} ({m})')
print('\n⚠️  Restart kernel after first install.')

  ✅ requests (uv)
  ✅ pandas (uv)
  ✅ sqlalchemy (uv)
  ✅ psycopg2-binary (uv)

⚠️  Restart kernel after first install.


In [1]:
# ── src path bootstrap + shared config (modules live in /src) ───────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from nb_support import *
init_env("local")   # default safe env; POLARIS_URL/REALM/ROOT_SECRET/MINIO_*/PG_* now set
# ============================================================
# Setup
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

POLARIS_URL = POLARIS_URL  # from init_env (src/config)
REALM       = REALM  # from init_env
PG_URL      = f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"  # from init_env
engine      = create_engine(PG_URL)

def get_token(client_id='root', client_secret=ROOT_SECRET):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={'grant_type': 'client_credentials', 'client_id': client_id,
              'client_secret': client_secret, 'scope': 'PRINCIPAL_ROLE:ALL'}
    )
    return r.json()['access_token']

def h(token):
    return {'Authorization': f'Bearer {token}', 'Polaris-Realm': REALM, 'Content-Type': 'application/json'}

def pg(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

def check(r, expected_ok=True):
    ok = r.status_code in [200, 201]
    icon = '✅' if ok == expected_ok else '❌ UNEXPECTED'
    label = 'ALLOWED' if ok else 'DENIED'
    print(f'  {icon} {label} ({r.status_code})')
    if not ok and r.text:
        err = r.json().get('error', {}).get('message', r.text[:100])
        print(f'     → {err}')
    return ok

TOKEN_ROOT = get_token()
print('✅ Connected')

✅ Connected


In [6]:
# ============================================================
# Rotate All Principals (delete + recreate, preserving roles)
# ============================================================
print('=== Rotating All Principal Credentials ===\n')

PRINCIPAL_CREDS = {}

# Skip root and service_admin — system principals
SKIP = {'root', 'service_admin'}

for p in principals:
    pname = p['name']
    if pname in SKIP:
        print(f'⏭️  Skipping: {pname}')
        continue

    # Save current principal roles
    pr_r = requests.get(
        f'{POLARIS_URL}/api/management/v1/principals/{pname}/principal-roles',
        headers=h(TOKEN_ROOT)
    )
    roles = [pr['name'] for pr in pr_r.json().get('roles', [])]

    # Delete
    requests.delete(
        f'{POLARIS_URL}/api/management/v1/principals/{pname}',
        headers=h(TOKEN_ROOT)
    )

    # Recreate
    r = requests.post(
        f'{POLARIS_URL}/api/management/v1/principals',
        headers=h(TOKEN_ROOT),
        json={'name': pname, 'type': 'SERVICE'}
    )
    creds = r.json().get('credentials', {})
    cid, csecret = creds['clientId'], creds['clientSecret']
    PRINCIPAL_CREDS[pname] = (cid, csecret)

    # Reassign roles
    for role in roles:
        requests.put(
            f'{POLARIS_URL}/api/management/v1/principals/{pname}/principal-roles',
            headers=h(TOKEN_ROOT),
            json={'principalRole': {'name': role}}
        )

    print(f'✅ {pname}')
    print(f'   clientId:     {cid}')
    print(f'   clientSecret: {csecret}')
    print(f'   roles:        {roles}')
    print()

# Get tokens for all rotated principals
TOKENS = {'root': TOKEN_ROOT}
for pname, (cid, csecret) in PRINCIPAL_CREDS.items():
    try:
        TOKENS[pname] = get_token(cid, csecret)
    except Exception as e:
        print(f'❌ Token failed: {pname} — {e}')

print(f'✅ Tokens ready: {list(TOKENS.keys())}')

=== Rotating All Principal Credentials ===

⏭️  Skipping: root
✅ spark-service
   clientId:     d59d8566843d7f4b
   clientSecret: 402d150459f36112b232b584f44d4851
   roles:        ['data-engineer']

✅ carol
   clientId:     e1e25e182c2db515
   clientSecret: 44f95e58fdeef122ec94d49b13bcb45d
   roles:        ['ml-engineer']

✅ bob
   clientId:     92cd36fd6151f1e3
   clientSecret: 955d27dbc917ec50f4015e9d84a3a445
   roles:        ['data-analyst']

✅ trino-service
   clientId:     b001db0be896ea4b
   clientSecret: 1a58fca25ec54a2f23afd77cd28208ee
   roles:        ['data-analyst']

✅ etl-service
   clientId:     a5e10251f14242b7
   clientSecret: 33f50233aa44cd4f371a9251f6d86a40
   roles:        ['data-engineer']

✅ alice
   clientId:     15d3e12c63c09e35
   clientSecret: 7af5cd59c62f6642357c786902ee2778
   roles:        ['ops-admin', 'data-engineer']

✅ Tokens ready: ['root', 'spark-service', 'carol', 'bob', 'trino-service', 'etl-service', 'alice']


## Step 1 — Full RBAC Chain Review
현재 설정된 전체 RBAC 체인을 확인합니다.

In [7]:
# Step 1 — Show full RBAC chain from PostgreSQL
print('=== Full RBAC Chain ===')

PRIVILEGE_CODES = {
    1:'CATALOG_MANAGE_ACCESS', 2:'CATALOG_MANAGE_CONTENT',
    3:'CATALOG_MANAGE_METADATA', 4:'CATALOG_READ_PROPERTIES',
    5:'CATALOG_WRITE_PROPERTIES', 6:'NAMESPACE_CREATE',
    7:'NAMESPACE_LIST', 8:'NAMESPACE_READ_PROPERTIES',
    9:'NAMESPACE_WRITE_PROPERTIES', 10:'NAMESPACE_FULL_METADATA',
    11:'SERVICE_MANAGE_ACCESS', 12:'PRINCIPAL_ROLE_USAGE',
    13:'CATALOG_ROLE_USAGE', 14:'TABLE_READ_DATA',
    15:'TABLE_WRITE_DATA', 16:'TABLE_READ_PROPERTIES',
    17:'TABLE_WRITE_PROPERTIES', 18:'TABLE_FULL_METADATA',
    19:'VIEW_READ_PROPERTIES', 20:'VIEW_WRITE_PROPERTIES',
    21:'VIEW_FULL_METADATA', 22:'VIEW_CREATE', 23:'TABLE_CREATE',
}
TYPE_CODES = {1:'ROOT', 2:'PRINCIPAL', 3:'PRINCIPAL_ROLE',
              4:'CATALOG', 5:'NAMESPACE', 6:'TABLE/VIEW', 7:'CATALOG_ROLE'}

# Get all principals and their role chains
principals = requests.get(f'{POLARIS_URL}/api/management/v1/principals', headers=h(TOKEN_ROOT)).json().get('principals', [])
catalogs   = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',   headers=h(TOKEN_ROOT)).json().get('catalogs', [])

for p in principals:
    pname = p['name']
    pr_r  = requests.get(f'{POLARIS_URL}/api/management/v1/principals/{pname}/principal-roles', headers=h(TOKEN_ROOT))
    print(f'\nPrincipal: {pname}')
    for pr in pr_r.json().get('roles', []):
        prname = pr['name']
        print(f'  └── Principal Role: {prname}')
        for c in catalogs:
            cname = c['name']
            cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles/{prname}/catalog-roles/{cname}', headers=h(TOKEN_ROOT))
            for cr in cr_r.json().get('roles', []):
                crname = cr['name']
                print(f'      └── Catalog Role: {crname} (on {cname})')
                g_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}/grants', headers=h(TOKEN_ROOT))
                for g in g_r.json().get('grants', []):
                    scope = g.get('tableName') or g.get('viewName') or g.get('namespace') or 'catalog'
                    print(f'          └── {g["privilege"]} [{g["type"]}:{scope}]')

=== Full RBAC Chain ===

Principal: root
  └── Principal Role: service_admin
      └── Catalog Role: catalog_admin (on test-catalog)
          └── CATALOG_MANAGE_ACCESS [catalog:catalog]
          └── CATALOG_MANAGE_METADATA [catalog:catalog]
      └── Catalog Role: catalog_admin (on ml-catalog)
          └── CATALOG_MANAGE_ACCESS [catalog:catalog]
          └── CATALOG_MANAGE_METADATA [catalog:catalog]
      └── Catalog Role: catalog_admin (on analytics-catalog)
          └── CATALOG_MANAGE_ACCESS [catalog:catalog]
          └── CATALOG_MANAGE_METADATA [catalog:catalog]

Principal: spark-service
  └── Principal Role: data-engineer
      └── Catalog Role: data-engineer-role (on test-catalog)
          └── CATALOG_MANAGE_CONTENT [catalog:catalog]
      └── Catalog Role: engineer-role (on analytics-catalog)
          └── CATALOG_MANAGE_CONTENT [catalog:catalog]

Principal: carol
  └── Principal Role: ml-engineer
      └── Catalog Role: ml-role (on ml-catalog)
          └── CATALOG_MANAGE

## Step 2 — Catalog Scope vs Namespace Scope vs Table Scope

핵심 이해:
- `TABLE_READ_DATA` at **catalog** scope = 카탈로그 내 **모든** 테이블 읽기 가능
- `TABLE_READ_DATA` at **namespace** scope = 해당 네임스페이스 내 **모든** 테이블 읽기 가능  
- `TABLE_READ_DATA` at **table** scope = **해당 테이블만** 읽기 가능
- Catalog scope ≠ 하위 권한 자동 부여 (TABLE_READ_DATA ≠ NAMESPACE_LIST)

In [3]:
# Step 2 — Show grants with scope breakdown
print('=== Grant Scope Analysis ===')

for c in catalogs:
    cname = c['name']
    cr_list = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN_ROOT)).json().get('roles', [])
    print(f'\nCatalog: {cname}')
    for cr in cr_list:
        crname = cr['name']
        grants = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}/grants', headers=h(TOKEN_ROOT)).json().get('grants', [])
        if not grants:
            continue
        print(f'  Catalog Role: {crname}')
        catalog_grants   = [g for g in grants if g['type'] == 'catalog']
        namespace_grants = [g for g in grants if g['type'] == 'namespace']
        table_grants     = [g for g in grants if g['type'] == 'table']
        view_grants      = [g for g in grants if g['type'] == 'view']
        if catalog_grants:
            print(f'    [catalog scope]   → applies to entire catalog')
            for g in catalog_grants:
                print(f'      {g["privilege"]}')
        if namespace_grants:
            print(f'    [namespace scope] → applies to specific namespace only')
            for g in namespace_grants:
                print(f'      {g["privilege"]} on namespace:{g["namespace"]}')
        if table_grants:
            print(f'    [table scope]     → applies to specific table only')
            for g in table_grants:
                print(f'      {g["privilege"]} on {g["namespace"]}.{g["tableName"]}')
        if view_grants:
            print(f'    [view scope]      → applies to specific view only')
            for g in view_grants:
                print(f'      {g["privilege"]} on {g["namespace"]}.{g["viewName"]}')

=== Grant Scope Analysis ===

Catalog: test-catalog
  Catalog Role: catalog_admin
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_ACCESS
      CATALOG_MANAGE_METADATA
  Catalog Role: data-engineer-role
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_CONTENT

Catalog: ml-catalog
  Catalog Role: catalog_admin
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_ACCESS
      CATALOG_MANAGE_METADATA
  Catalog Role: admin-role
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_ACCESS
      CATALOG_MANAGE_CONTENT
  Catalog Role: ml-role
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_CONTENT
    [namespace scope] → applies to specific namespace only
      NAMESPACE_FULL_METADATA on namespace:['features']
    [table scope]     → applies to specific table only
      TABLE_WRITE_DATA on ['features'].user_features

Catalog: analytics-catalog
  Catalog Role: analyst-role
    [catalog scope] 

## Step 3 — Access Verification per Principal

각 Principal이 실제로 무엇을 할 수 있는지 API 호출로 확인합니다.

**bob (data-analyst)**:
- `CATALOG_READ_PROPERTIES` at catalog → 카탈로그 읽기 ✅
- `TABLE_READ_DATA` at catalog → 모든 테이블 읽기 ✅
- `NAMESPACE_FULL_METADATA` on reporting → reporting 네임스페이스 ✅
- ml-catalog → 권한 없음 ❌

In [8]:
# Step 3 — Get tokens for each principal
# First check what principals we have and get their credentials from PostgreSQL
print('=== Principal Credentials (from PostgreSQL) ===')
df = pg("""
    SELECT name, internal_properties->>'client_id' AS client_id
    FROM polaris_schema.entities
    WHERE type_code = 2
    ORDER BY name
""")
display(df)
print('\n💡 client_secret is not stored — use credentials saved at creation time')
print('   Or rotate credentials to get new ones')

=== Principal Credentials (from PostgreSQL) ===


,name,client_id
0,alice,15d3e12c63c09e35
1,bob,92cd36fd6151f1e3
2,carol,e1e25e182c2db515
3,etl-service,a5e10251f14242b7
4,root,root
5,spark-service,d59d8566843d7f4b
6,trino-service,b001db0be896ea4b



💡 client_secret is not stored — use credentials saved at creation time
   Or rotate credentials to get new ones


In [9]:
# Step 3b — Set credentials here (replace with actual values)
# ============================================================
# Replace these with actual credentials saved when principals were created
# Or rotate: POST /api/management/v1/principals/{name}/rotate
# ============================================================

# Get tokens for available principals
TOKENS = {'root': TOKEN_ROOT}
for pname, (cid, csecret) in PRINCIPAL_CREDS.items():
    try:
        TOKENS[pname] = get_token(cid, csecret)
        print(f'✅ Token obtained: {pname}')
    except Exception as e:
        print(f'❌ Failed: {pname} — {e}')

print(f'\nAvailable tokens: {list(TOKENS.keys())}')

✅ Token obtained: spark-service
✅ Token obtained: carol
✅ Token obtained: bob
✅ Token obtained: trino-service
✅ Token obtained: etl-service
✅ Token obtained: alice

Available tokens: ['root', 'spark-service', 'carol', 'bob', 'trino-service', 'etl-service', 'alice']


In [10]:
# Step 3c — Test access for each available principal
def test_access(principal_name, token):
    print(f'\n{"="*50}')
    print(f'Principal: {principal_name}')
    print(f'{"="*50}')

    # Test 1: List analytics-catalog namespaces
    print('\n1. List analytics-catalog namespaces:')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces', headers=h(token))
    if check(r):
        ns = [n[0] if isinstance(n, list) else n for n in r.json().get('namespaces', [])]
        print(f'     namespaces: {ns}')

    # Test 2: List tables in raw namespace
    print('\n2. List tables in analytics-catalog.raw:')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/raw/tables', headers=h(token))
    if check(r):
        tables = [t.get('name') for t in r.json().get('identifiers', [])]
        print(f'     tables: {tables}')

    # Test 3: Load table metadata (raw.events)
    print('\n3. Load table metadata (raw.events):')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/raw/tables/events', headers=h(token))
    check(r)
 
    # Test 4: List tables in reporting namespace
    print('\n4. List tables in analytics-catalog.reporting:')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/reporting/tables', headers=h(token))
    if check(r):
        tables = [t.get('name') for t in r.json().get('identifiers', [])]
        print(f'     tables: {tables}')

    # Test 5: List views in reporting namespace
    print('\n5. List views in analytics-catalog.reporting:')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/reporting/views', headers=h(token))
    if check(r):
        views = [v.get('name') for v in r.json().get('identifiers', [])]
        print(f'     views: {views}')

    # Test 6: Access ml-catalog (should fail for non-ml principals)
    print('\n6. List ml-catalog namespaces:')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/ml-catalog/namespaces', headers=h(token))
    check(r)

    # Test 7: Load ml-catalog table (should fail)
    print('\n7. Load ml table (features.user_features):')
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/ml-catalog/namespaces/features/tables/user_features', headers=h(token))
    check(r)

# Run tests for all available principals
for pname, token in TOKENS.items():
    test_access(pname, token)


Principal: root

1. List analytics-catalog namespaces:
  ✅ ALLOWED (200)
     namespaces: ['raw', 'processed', 'reporting']

2. List tables in analytics-catalog.raw:
  ✅ ALLOWED (200)
     tables: ['users', 'events']

3. Load table metadata (raw.events):
  ✅ ALLOWED (200)

4. List tables in analytics-catalog.reporting:
  ✅ ALLOWED (200)
     tables: ['monthly_revenue']

5. List views in analytics-catalog.reporting:
  ✅ ALLOWED (200)
     views: ['weekly_active_users']

6. List ml-catalog namespaces:
  ✅ ALLOWED (200)

7. Load ml table (features.user_features):
  ✅ ALLOWED (200)

Principal: spark-service

1. List analytics-catalog namespaces:
  ✅ ALLOWED (200)
     namespaces: ['raw', 'processed', 'reporting']

2. List tables in analytics-catalog.raw:
  ✅ ALLOWED (200)
     tables: ['users', 'events']

3. Load table metadata (raw.events):
  ✅ ALLOWED (200)

4. List tables in analytics-catalog.reporting:
  ✅ ALLOWED (200)
     tables: ['monthly_revenue']

5. List views in analytics-cata

## Step 4 — Privilege Scope Proof

`TABLE_READ_DATA` at catalog scope vs namespace scope vs table scope 동작 차이를 직접 확인합니다.

In [11]:
# Step 4 — Create a test role with specific scope and verify behavior
print('=== Privilege Scope Test ===')
print('Creating test-role with TABLE_READ_DATA at TABLE scope only (raw.events)\n')

# Create a test catalog role
r = requests.post(
    f'{POLARIS_URL}/api/management/v1/catalogs/analytics-catalog/catalog-roles',
    headers=h(TOKEN_ROOT),
    json={'catalogRole': {'name': 'scope-test-role'}}
)
print(f'Create scope-test-role: {r.status_code}')

# Grant TABLE_READ_DATA at TABLE scope only
r = requests.put(
    f'{POLARIS_URL}/api/management/v1/catalogs/analytics-catalog/catalog-roles/scope-test-role/grants',
    headers=h(TOKEN_ROOT),
    json={'grant': {'type': 'table', 'namespace': ['raw'], 'tableName': 'events', 'privilege': 'TABLE_READ_DATA'}}
)
print(f'Grant TABLE_READ_DATA on raw.events: {r.status_code}')

# Create a test principal role and assign scope-test-role
r = requests.post(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN_ROOT),
                  json={'principalRole': {'name': 'scope-test-pr'}})
print(f'Create scope-test-pr: {r.status_code}')

r = requests.put(
    f'{POLARIS_URL}/api/management/v1/principal-roles/scope-test-pr/catalog-roles/analytics-catalog',
    headers=h(TOKEN_ROOT),
    json={'catalogRole': {'name': 'scope-test-role'}}
)
print(f'Assign scope-test-role to scope-test-pr: {r.status_code}')

print('\n✅ Test role created with TABLE_READ_DATA on raw.events ONLY')
print('Expected behavior:')
print('  raw.events      → ALLOWED (explicit table grant)')
print('  raw.users       → DENIED  (different table, no grant)')
print('  reporting.*     → DENIED  (different namespace, no grant)')
print('  ml-catalog.*    → DENIED  (different catalog, no grant)')

=== Privilege Scope Test ===
Creating test-role with TABLE_READ_DATA at TABLE scope only (raw.events)

Create scope-test-role: 201
Grant TABLE_READ_DATA on raw.events: 201
Create scope-test-pr: 201
Assign scope-test-role to scope-test-pr: 201

✅ Test role created with TABLE_READ_DATA on raw.events ONLY
Expected behavior:
  raw.events      → ALLOWED (explicit table grant)
  raw.users       → DENIED  (different table, no grant)
  reporting.*     → DENIED  (different namespace, no grant)
  ml-catalog.*    → DENIED  (different catalog, no grant)


In [12]:
# Step 4b — Create test principal and get token
r = requests.post(f'{POLARIS_URL}/api/management/v1/principals', headers=h(TOKEN_ROOT),
                  json={'name': 'scope-test-user', 'type': 'SERVICE'})
print(f'Create scope-test-user: {r.status_code}')
creds = r.json().get('credentials', {})
TEST_CLIENT_ID     = creds.get('clientId')
TEST_CLIENT_SECRET = creds.get('clientSecret')
print(f'clientId: {TEST_CLIENT_ID}')

# Assign principal role
r = requests.put(
    f'{POLARIS_URL}/api/management/v1/principals/scope-test-user/principal-roles',
    headers=h(TOKEN_ROOT),
    json={'principalRole': {'name': 'scope-test-pr'}}
)
print(f'Assign scope-test-pr to scope-test-user: {r.status_code}')

# Get token
TOKEN_TEST = get_token(TEST_CLIENT_ID, TEST_CLIENT_SECRET)
print(f'\n✅ scope-test-user token obtained')

Create scope-test-user: 201
clientId: e9cb67634b283ef1
Assign scope-test-pr to scope-test-user: 201

✅ scope-test-user token obtained


In [13]:
# Step 4c — Verify scope behavior
print('=== Scope Test Results ===')
print('scope-test-user has TABLE_READ_DATA on raw.events ONLY\n')

tests = [
    ('List namespaces in analytics-catalog',
     'GET', f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces',
     False),  # No NAMESPACE_LIST granted

    ('Load raw.events metadata (table scope grant)',
     'GET', f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/raw/tables/events',
     True),   # Should be allowed

    ('Load raw.users metadata (no grant for this table)',
     'GET', f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/raw/tables/users',
     False),  # Should be denied

    ('List tables in raw namespace',
     'GET', f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/raw/tables',
     False),  # No TABLE_LIST granted

    ('Load reporting.monthly_revenue (different namespace)',
     'GET', f'{POLARIS_URL}/api/catalog/v1/analytics-catalog/namespaces/reporting/tables/monthly_revenue',
     False),  # Should be denied

    ('List ml-catalog namespaces (different catalog)',
     'GET', f'{POLARIS_URL}/api/catalog/v1/ml-catalog/namespaces',
     False),  # Should be denied
]

for description, method, url, expected_ok in tests:
    print(f'\nTest: {description}')
    print(f'  Expected: {"ALLOWED" if expected_ok else "DENIED"}')
    r = requests.get(url, headers=h(TOKEN_TEST))
    check(r, expected_ok)

=== Scope Test Results ===
scope-test-user has TABLE_READ_DATA on raw.events ONLY


Test: List namespaces in analytics-catalog
  Expected: DENIED
  ✅ DENIED (403)
     → Principal 'scope-test-user' with activated PrincipalRoles '[scope-test-pr]' and activated grants via '[scope-test-pr, scope-test-role]' is not authorized for op LIST_NAMESPACES

Test: Load raw.events metadata (table scope grant)
  Expected: ALLOWED
  ✅ ALLOWED (200)

Test: Load raw.users metadata (no grant for this table)
  Expected: DENIED
  ✅ DENIED (403)
     → Principal 'scope-test-user' with activated PrincipalRoles '[scope-test-pr]' and activated grants via '[scope-test-pr, scope-test-role]' is not authorized for op LOAD_TABLE

Test: List tables in raw namespace
  Expected: DENIED
  ✅ DENIED (403)
     → Principal 'scope-test-user' with activated PrincipalRoles '[scope-test-pr]' and activated grants via '[scope-test-pr, scope-test-role]' is not authorized for op LIST_TABLES

Test: Load reporting.monthly_revenue (

In [14]:
# Step 4d — Cleanup test entities
print('=== Cleanup ===')

r = requests.delete(f'{POLARIS_URL}/api/management/v1/principals/scope-test-user', headers=h(TOKEN_ROOT))
print(f'Delete scope-test-user: {r.status_code}')

r = requests.delete(f'{POLARIS_URL}/api/management/v1/principal-roles/scope-test-pr', headers=h(TOKEN_ROOT))
print(f'Delete scope-test-pr: {r.status_code}')

r = requests.delete(f'{POLARIS_URL}/api/management/v1/catalogs/analytics-catalog/catalog-roles/scope-test-role', headers=h(TOKEN_ROOT))
print(f'Delete scope-test-role: {r.status_code}')

print('\n✅ Cleanup complete')

=== Cleanup ===
Delete scope-test-user: 204
Delete scope-test-pr: 204
Delete scope-test-role: 204

✅ Cleanup complete


## Step 5 — Grant Resolution Summary (from PostgreSQL)

grant_records 테이블에서 실제 권한 부여 현황을 확인합니다.

In [15]:
# Step 5 — Full grant matrix from PostgreSQL
print('=== Grant Resolution Matrix (PostgreSQL) ===')

PRIVILEGE_CODES = {
    1:'CATALOG_MANAGE_ACCESS', 2:'CATALOG_MANAGE_CONTENT',
    3:'CATALOG_MANAGE_METADATA', 4:'CATALOG_READ_PROPERTIES',
    5:'CATALOG_WRITE_PROPERTIES', 6:'NAMESPACE_CREATE',
    7:'NAMESPACE_LIST', 8:'NAMESPACE_READ_PROPERTIES',
    9:'NAMESPACE_WRITE_PROPERTIES', 10:'NAMESPACE_FULL_METADATA',
    11:'SERVICE_MANAGE_ACCESS', 12:'PRINCIPAL_ROLE_USAGE',
    13:'CATALOG_ROLE_USAGE', 14:'TABLE_READ_DATA',
    15:'TABLE_WRITE_DATA', 16:'TABLE_READ_PROPERTIES',
    17:'TABLE_WRITE_PROPERTIES', 18:'TABLE_FULL_METADATA',
    19:'VIEW_READ_PROPERTIES', 20:'VIEW_WRITE_PROPERTIES',
    21:'VIEW_FULL_METADATA', 22:'VIEW_CREATE', 23:'TABLE_CREATE',
}

df = pg("""
    SELECT
        g.privilege_code,
        grantee.name   AS grantee,
        grantee.type_code AS grantee_type,
        securable.name AS securable,
        securable.type_code AS securable_type
    FROM polaris_schema.grant_records g
    JOIN polaris_schema.entities grantee
        ON g.grantee_id = grantee.id AND g.realm_id = grantee.realm_id
    JOIN polaris_schema.entities securable
        ON g.securable_id = securable.id AND g.realm_id = securable.realm_id
    WHERE grantee.type_code = 7  -- catalog roles only
    ORDER BY securable.name, grantee.name, g.privilege_code
""")

TYPE_CODES = {1:'ROOT', 2:'PRINCIPAL', 3:'PRINCIPAL_ROLE',
              4:'CATALOG', 5:'NAMESPACE', 6:'TABLE/VIEW', 7:'CATALOG_ROLE'}

df['privilege']      = df['privilege_code'].map(PRIVILEGE_CODES)
df['grantee_type']   = df['grantee_type'].map(TYPE_CODES)
df['securable_type'] = df['securable_type'].map(TYPE_CODES)

print('\nAll catalog role grants:')
display(df[['grantee', 'privilege', 'securable', 'securable_type']].rename(
    columns={'grantee': 'catalog_role', 'securable': 'target', 'securable_type': 'target_type'}
))

print('\n💡 Reading: catalog_role has privilege on target')
print('   securable_type=CATALOG  → catalog scope (applies to all inside)')
print('   securable_type=NAMESPACE → namespace scope (applies to all inside)')
print('   securable_type=TABLE/VIEW → specific object only')

=== Grant Resolution Matrix (PostgreSQL) ===

All catalog role grants:


,catalog_role,privilege,target,target_type



💡 Reading: catalog_role has privilege on target
   securable_type=CATALOG  → catalog scope (applies to all inside)
   securable_type=NAMESPACE → namespace scope (applies to all inside)
   securable_type=TABLE/VIEW → specific object only


## 관찰 결과 (Observations)

- Catalog scope grant → 카탈로그 내 모든 해당 타입 객체에 적용됨
- Table scope grant → 해당 테이블만 적용, 같은 namespace의 다른 테이블은 접근 불가
- TABLE_READ_DATA ≠ NAMESPACE_LIST: 서로 다른 권한, 상속되지 않음
- Principal Role에는 직접 권한 부여 불가 — 반드시 Catalog Role 경유
- grant_records 테이블이 모든 RBAC 권한의 실제 저장소

## 핵심 학습 포인트 (Key Takeaways)

1. Scope = 권한이 적용되는 범위 (catalog/namespace/table/view)
2. Privilege = 허용되는 작업 (TABLE_READ_DATA, NAMESPACE_LIST 등)
3. Scope와 Privilege는 독립적 — catalog scope라도 해당 privilege만 허용
4. 권한 체인: Privilege → Catalog Role → Principal Role → Principal
5. Grant resolution: 요청 시 위 체인을 역순으로 탐색하여 허용/거부 결정